# Build and Clean Integrated CBSA Quarterly Panel

This notebook combines the build and clean scripts to produce `firmscape_integrated_cbsa_quarterly_cleaned.csv` from the four original CSVs.

**Inputs:**
- `hpi_at_metro.csv` - FHFA HPI by metro/quarter
- `cleaned_zillow_data.csv` - Zillow monthly price data
- `companies_us_100plus_clean.csv` - Firms with city, state, founded, industry
- `kaggle_companies_cleaned.csv` - Firms with locality, year founded, employee estimates

**Outputs:**
- `data/firmscape_integrated_cbsa_quarterly.csv` - Integrated panel (before cleaning)
- `data/firmscape_integrated_cbsa_quarterly_cleaned.csv` - Final cleaned panel

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Paths: inputs in DATA_DIR by default; set FIRMSCAPE_INPUT_DIR to override
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = Path(os.environ.get("FIRMSCAPE_INPUT_DIR", str(DATA_DIR)))

HPI_PATH = INPUT_DIR / "hpi_at_metro.csv"
ZILLOW_PATH = INPUT_DIR / "cleaned_zillow_data.csv"
COMPANIES_US_PATH = INPUT_DIR / "companies_us_100plus_clean.csv"
KAGGLE_PATH = INPUT_DIR / "kaggle_companies_cleaned.csv"

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {DATA_DIR}")

Input directory: /Users/rishil/Desktop/FirmScape/data
Output directory: /Users/rishil/Desktop/FirmScape/data


In [9]:
# State name to abbreviation mapping
STATE_TO_ABBR = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT",
    "delaware": "DE", "district of columbia": "DC", "florida": "FL",
    "georgia": "GA", "hawaii": "HI", "idaho": "ID", "illinois": "IL",
    "indiana": "IN", "iowa": "IA", "kansas": "KS", "kentucky": "KY",
    "louisiana": "LA", "maine": "ME", "maryland": "MD", "massachusetts": "MA",
    "michigan": "MI", "minnesota": "MN", "mississippi": "MS", "missouri": "MO",
    "montana": "MT", "nebraska": "NE", "nevada": "NV", "new hampshire": "NH",
    "new jersey": "NJ", "new mexico": "NM", "new york": "NY", "north carolina": "NC",
    "north dakota": "ND", "ohio": "OH", "oklahoma": "OK", "oregon": "OR",
    "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA", "west virginia": "WV",
    "wisconsin": "WI", "wyoming": "WY",
}

## Helper Functions

In [10]:
def _size_to_mid(s: str) -> float:
    """Map size range to midpoint (for median_size_mid)."""
    if pd.isna(s) or not isinstance(s, str):
        return np.nan
    s = s.strip().upper()
    if "10K+" in s or "10001+" in s:
        return 15000.0
    if "1K-5K" in s or "1001-5000" in s:
        return 3000.0
    if "201-500" in s or "501-1000" in s:
        return 500.0
    if "51-200" in s:
        return 125.5
    if "11-50" in s:
        return 30.0
    if "1-10" in s:
        return 5.0
    return np.nan

def _is_large_size(s: str) -> bool:
    """True if size is 1K+ (for share_large_proxy)."""
    if pd.isna(s) or not isinstance(s, str):
        return False
    s = s.strip().upper()
    return "1K" in s or "10K" in s or "10001+" in s or "1001" in s

## Step 1: Load and Clean HPI Data

In [11]:
print("Loading HPI...")
hpi = pd.read_csv(HPI_PATH, header=None, names=["metro_name", "cbsa_code", "year", "quarter", "hpi_value", "hpi_se"])
hpi["fhfa_index"] = pd.to_numeric(hpi["hpi_value"].replace("-", np.nan).replace("", np.nan), errors="coerce")
hpi["cbsa_code"] = pd.to_numeric(hpi["cbsa_code"], errors="coerce")
hpi["year"] = pd.to_numeric(hpi["year"], errors="coerce").astype("Int64")
hpi["quarter"] = pd.to_numeric(hpi["quarter"], errors="coerce").astype("Int64")
hpi = hpi.drop(columns=["hpi_value", "hpi_se"], errors="ignore")
hpi = hpi.dropna(subset=["cbsa_code", "year", "quarter"])

# Add growth rates
hpi = hpi.sort_values(["cbsa_code", "year", "quarter"])
hpi["fhfa_qoq"] = hpi.groupby("cbsa_code")["fhfa_index"].pct_change(1, fill_method=None)
hpi["fhfa_yoy"] = hpi.groupby("cbsa_code")["fhfa_index"].pct_change(4, fill_method=None)

metro_cbsa = hpi[["metro_name", "cbsa_code"]].drop_duplicates()
print(f"Loaded {len(hpi)} HPI rows for {len(metro_cbsa)} metros")
hpi.head()

Loading HPI...
Loaded 83230 HPI rows for 410 metros


,metro_name,cbsa_code,year,quarter,fhfa_index,fhfa_qoq,fhfa_yoy
0,"Abilene, TX",10180,1975,1,NaN,NaN,NaN
1,"Abilene, TX",10180,1975,2,NaN,NaN,NaN
2,"Abilene, TX",10180,1975,3,NaN,NaN,NaN
3,"Abilene, TX",10180,1975,4,NaN,NaN,NaN
4,"Abilene, TX",10180,1976,1,NaN,NaN,NaN


## Step 2: Load and Aggregate Zillow Data

In [12]:
print("Loading Zillow...")
zillow = pd.read_csv(ZILLOW_PATH)
zillow["date"] = pd.to_datetime(zillow["date"], errors="coerce")
zillow = zillow.dropna(subset=["date", "price"])
zillow["year"] = zillow["date"].dt.year
zillow["quarter"] = zillow["date"].dt.quarter
zillow_q = zillow.groupby(["RegionName", "year", "quarter"], as_index=False)["price"].mean()
zillow_q = zillow_q.rename(columns={"price": "zillow_price_q"})
print(f"Loaded {len(zillow_q)} Zillow quarterly records")
zillow_q.head()

Loading Zillow...
Loaded 76924 Zillow quarterly records


,RegionName,year,quarter,zillow_price_q
0,"Aberdeen, SD",2009,1,126015.606181
1,"Aberdeen, SD",2009,2,125694.679015
2,"Aberdeen, SD",2009,3,125187.232900
3,"Aberdeen, SD",2009,4,124856.726130
4,"Aberdeen, SD",2010,1,124882.177157


## Step 3: Load Companies Data and Assign CBSA Codes

In [ ]:
print("Loading companies_us and assigning CBSA...")
companies = pd.read_csv(COMPANIES_US_PATH)
companies = companies[companies["country_code"].astype(str).str.upper().eq("US")].copy()

def to_abbr(s):
    s = str(s).strip()
    if len(s) == 2:
        return s.upper()
    return STATE_TO_ABBR.get(s.lower(), "")

companies["state_abbr"] = companies["state"].astype(str).apply(to_abbr)
companies["city_norm"] = companies["city"].astype(str).str.strip()
companies["metro_key"] = companies["city_norm"] + ", " + companies["state_abbr"]
mc = metro_cbsa[["metro_name", "cbsa_code"]].drop_duplicates()
companies = companies.merge(mc, left_on="metro_key", right_on="metro_name", how="left")
companies = companies.drop(columns=["metro_key", "state_abbr", "city_norm", "metro_name"], errors="ignore")
companies["founded"] = pd.to_numeric(companies["founded"], errors="coerce")
companies = companies.dropna(subset=["cbsa_code"])
print(f"Loaded {len(companies)} companies with CBSA codes")
companies.head()

Loading companies_us and assigning CBSA...
Loaded 27124 companies with CBSA codes


,name,industry,size,type,founded,city,state,country_code,cbsa_code
9,2INgage,philanthropic fundraising services,51-200,Nonprofit,2018,Abilene,Texas,US,10180.0
26,All About e-Gift Rewards,it services and it consulting,51-200,Privately Held,2021,Dover,Delaware,US,20100.0
31,AB Powers,engines and power transmission equipment manuf...,51-200,Privately Held,2016,El Paso,Texas,US,21340.0
35,ABC Medicare Plans,insurance,201-500,Privately Held,2011,Oklahoma City,Oklahoma,US,36420.0
42,"Accede Mold & Tool Co., Inc.",plastics manufacturing,51-200,Privately Held,1981,Rochester,New York,US,40380.0


## Step 4: Compute Metro-Level Stock Metrics

In [14]:
# Compute stock metrics (constant per metro)
c = companies.copy()
c["size_mid"] = c["size"].astype(str).apply(_size_to_mid)
c["is_large"] = c["size"].astype(str).apply(_is_large_size)
stock_df = c.groupby("cbsa_code").agg(
    firm_count_total=("name", "count"),
    industry_list=("industry", list),
    share_large_proxy=("is_large", "mean"),
    median_size_mid=("size_mid", "median"),
).reset_index()

def hhi_top(ind_list):
    flat = [i for i in ind_list if isinstance(i, str)]
    if not flat:
        return np.nan, np.nan, 0
    vc = pd.Series(flat).value_counts()
    n = len(vc)
    shares = vc / vc.sum()
    hhi = (shares ** 2).sum()
    top = shares.iloc[0]
    return hhi, top, n

stock_df["hhi_stock"] = stock_df["industry_list"].apply(lambda x: hhi_top(x)[0])
stock_df["top_industry_share_stock"] = stock_df["industry_list"].apply(lambda x: hhi_top(x)[1])
stock_df["industry_count_stock"] = stock_df["industry_list"].apply(lambda x: hhi_top(x)[2])
stock_df = stock_df.drop(columns=["industry_list"])
print(f"Computed stock metrics for {len(stock_df)} metros")
stock_df.head()

Computed stock metrics for 212 metros


,cbsa_code,firm_count_total,share_large_proxy,median_size_mid,hhi_stock,top_industry_share_stock,industry_count_stock
0,10180.0,52,0.096154,125.5,0.047337,0.115385,32
1,10420.0,221,0.113122,125.5,0.019471,0.054299,89
2,10500.0,39,0.025641,125.5,0.054017,0.131579,26
3,10540.0,19,0.052632,125.5,0.074792,0.105263,15
4,10740.0,413,0.084746,125.5,0.027747,0.107056,108


## Step 5: Aggregate Firms by CBSA-Quarter (Flow Metrics)

In [15]:
# Aggregate firms by quarter (new firms founded in that quarter)
companies_flow = companies.copy()
companies_flow["founded"] = pd.to_numeric(companies_flow["founded"], errors="coerce")
companies_flow = companies_flow.dropna(subset=["founded"])
companies_flow["year"] = companies_flow["founded"].astype(int)
companies_flow["quarter"] = 1  # Assign all to Q1

firms_agg = companies_flow.groupby(["cbsa_code", "year", "quarter"]).agg(
    firms_founded=("name", "count"),
    industry_list=("industry", list),
).reset_index()

def hhi_top(ind_list):
    flat = [i for i in ind_list if isinstance(i, str)]
    if not flat:
        return np.nan, np.nan, 0
    vc = pd.Series(flat).value_counts()
    n = len(vc)
    shares = vc / vc.sum()
    hhi = (shares ** 2).sum()
    top = shares.iloc[0]
    return hhi, top, n

firms_agg["hhi_new"] = firms_agg["industry_list"].apply(lambda x: hhi_top(x)[0])
firms_agg["top_industry_share_new"] = firms_agg["industry_list"].apply(lambda x: hhi_top(x)[1])
firms_agg["industry_count_new"] = firms_agg["industry_list"].apply(lambda x: hhi_top(x)[2])
firms_agg = firms_agg.drop(columns=["industry_list"])
firms_agg["firms_founded_yoy"] = firms_agg.groupby("cbsa_code")["firms_founded"].pct_change(4, fill_method=None)
print(f"Aggregated firms into {len(firms_agg)} CBSA-quarter records")
firms_agg.head()

Aggregated firms into 11852 CBSA-quarter records


,cbsa_code,year,quarter,firms_founded,hhi_new,top_industry_share_new,industry_count_new,firms_founded_yoy
0,10180.0,1881,1,1,1.0,1.0,1,NaN
1,10180.0,1890,1,1,1.0,1.0,1,NaN
2,10180.0,1891,1,1,1.0,1.0,1,NaN
3,10180.0,1923,1,1,1.0,1.0,1,NaN
4,10180.0,1924,1,2,0.5,0.5,2,1.0


## Step 6: Load Kaggle Data and Aggregate Entry Employment

In [16]:
# Entry employment (Kaggle) removed from pipeline
pass

Loading Kaggle and assigning CBSA...
Aggregated entry employment into 0 CBSA-quarter records


,cbsa_code,year,quarter,entry_emp_proxy,entry_emp_firm_count,entry_emp_mean,entry_emp_median,entry_emp_yoy


## Step 7: Build Skeleton and Merge All Data

In [17]:
print("Building skeleton and merging...")
# Skeleton from HPI
skel = hpi[["cbsa_code", "metro_name", "year", "quarter", "fhfa_index", "fhfa_qoq", "fhfa_yoy"]].copy()
skel["city_state"] = skel["metro_name"]

# Merge Zillow (match RegionName to metro_name)
zq = zillow_q.merge(metro_cbsa[["metro_name", "cbsa_code"]].drop_duplicates(), 
                    left_on="RegionName", right_on="metro_name", how="inner")
zq = zq.groupby(["cbsa_code", "year", "quarter"], as_index=False)["zillow_price_q"].mean()
zq["zillow_qoq"] = zq.groupby("cbsa_code")["zillow_price_q"].pct_change(1, fill_method=None)
zq["zillow_yoy"] = zq.groupby("cbsa_code")["zillow_price_q"].pct_change(4, fill_method=None)
skel = skel.merge(zq, on=["cbsa_code", "year", "quarter"], how="left")

# Merge firms flow metrics
skel = skel.merge(firms_agg, on=["cbsa_code", "year", "quarter"], how="left")

# Merge stock metrics (constant per metro)
skel = skel.merge(stock_df, on="cbsa_code", how="left")

skel["yq"] = skel["year"].astype(str) + "Q" + skel["quarter"].astype(str)
print(f"Merged skeleton: {len(skel)} rows")
skel.head()

Building skeleton and merging...
Merged skeleton: 83230 rows


,cbsa_code,metro_name,year,quarter,fhfa_index,fhfa_qoq,fhfa_yoy,city_state,zillow_price_q,zillow_qoq,...,entry_emp_mean,entry_emp_median,entry_emp_yoy,firm_count_total,share_large_proxy,median_size_mid,hhi_stock,top_industry_share_stock,industry_count_stock,yq
0,10180,"Abilene, TX",1975,1,NaN,NaN,NaN,"Abilene, TX",NaN,NaN,...,NaN,NaN,NaN,52.0,0.096154,125.5,0.047337,0.115385,32.0,1975Q1
1,10180,"Abilene, TX",1975,2,NaN,NaN,NaN,"Abilene, TX",NaN,NaN,...,NaN,NaN,NaN,52.0,0.096154,125.5,0.047337,0.115385,32.0,1975Q2
2,10180,"Abilene, TX",1975,3,NaN,NaN,NaN,"Abilene, TX",NaN,NaN,...,NaN,NaN,NaN,52.0,0.096154,125.5,0.047337,0.115385,32.0,1975Q3
3,10180,"Abilene, TX",1975,4,NaN,NaN,NaN,"Abilene, TX",NaN,NaN,...,NaN,NaN,NaN,52.0,0.096154,125.5,0.047337,0.115385,32.0,1975Q4
4,10180,"Abilene, TX",1976,1,NaN,NaN,NaN,"Abilene, TX",NaN,NaN,...,NaN,NaN,NaN,52.0,0.096154,125.5,0.047337,0.115385,32.0,1976Q1


### What's in the final merged table?

At this point, the merged table (`skel`) has **one row per metro × quarter**, with:

**Key fields:**

- **Geography & time:**
  - `cbsa_code`, `metro_name`, `city_state` — metro identity
  - `year`, `quarter` — time index (we'll add `yq` and `date` later)

- **Housing signals:**
  - `fhfa_index` — FHFA HPI level
  - `fhfa_qoq`, `fhfa_yoy` — FHFA quarter-over-quarter and year-over-year growth
  - `zillow_price_q` — quarterly mean Zillow price level
  - `zillow_qoq`, `zillow_yoy` — Zillow growth rates

- **Firm entry (flow metrics):**
  - `firms_founded` — count of firms founded in that metro × year (assigned to Q1)
  - `firms_founded_yoy` — year-over-year growth in firm formation
  - `hhi_new`, `top_industry_share_new`, `industry_count_new` — industry diversity/concentration among **new firms** in that quarter

- **Metro stock structure (constant per metro):**
  - `firm_count_total` — total firms mapped to that metro
  - `hhi_stock`, `top_industry_share_stock`, `industry_count_stock` — industry concentration/diversity across **all firms** in the metro
  - `share_large_proxy`, `median_size_mid` — size proxies

- **Entry employment (from Kaggle):**
  - `entry_emp_proxy` — sum of employee estimates for new firms
  - `entry_emp_yoy` — year-over-year growth in entry employment
  - `entry_emp_firm_count`, `entry_emp_mean`, `entry_emp_median` — additional scale metrics

**What this enables:**
- Each row represents one metro in one quarter, with housing levels/growth, firm formation dynamics, industry mix, and employment proxies
- This structure supports time-series analysis, lagged relationships, and metro-level scoring (which we'll do in the next notebook)

## Step 8: Add Lags and Forward Variables

In [ ]:
# Add lagged and forward variables
out = skel.sort_values(["cbsa_code", "year", "quarter"])

for lag in [1, 2, 4, 8]:
    out[f"firms_founded_yoy_lag{lag}q"] = out.groupby("cbsa_code")["firms_founded_yoy"].shift(lag)
    out[f"hhi_new_lag{lag}q"] = out.groupby("cbsa_code")["hhi_new"].shift(lag)

out["fhfa_yoy_fwd4q"] = out.groupby("cbsa_code")["fhfa_yoy"].shift(-4)
out["zillow_yoy_fwd4q"] = out.groupby("cbsa_code")["zillow_yoy"].shift(-4)

# Ensure stock metrics are present
if "share_large_proxy" not in out.columns:
    out["share_large_proxy"] = np.nan
if "median_size_mid" not in out.columns:
    out["median_size_mid"] = np.nan

out["entry_emp_nonmissing"] = out["entry_emp_proxy"].notna().astype(float)
out["entry_emp_coverage"] = 1.0

print("Added lags and forward variables")

Added lags and forward variables


## Step 9: Export Integrated Panel

In [ ]:
# Reorder columns to match expected schema
cols = [
    "cbsa_code", "metro_name", "city_state", "year", "quarter",
    "fhfa_index", "zillow_price_q", "fhfa_qoq", "fhfa_yoy", "zillow_qoq", "zillow_yoy",
    "firms_founded", "firms_founded_yoy", "hhi_new", "top_industry_share_new", "industry_count_new",
    "firm_count_total", "hhi_stock", "top_industry_share_stock", "industry_count_stock",
    "share_large_proxy", "median_size_mid",
    "firms_founded_yoy_lag1q", "hhi_new_lag1q", "firms_founded_yoy_lag2q", "hhi_new_lag2q",
    "firms_founded_yoy_lag4q", "hhi_new_lag4q", "firms_founded_yoy_lag8q", "hhi_new_lag8q",
    "fhfa_yoy_fwd4q", "zillow_yoy_fwd4q", "yq",
    "entry_emp_proxy", "entry_emp_yoy", "entry_emp_firm_count", "entry_emp_mean", "entry_emp_median",
    "entry_emp_nonmissing", "entry_emp_coverage",
    "entry_emp_yoy_lag1q", "entry_emp_yoy_lag2q", "entry_emp_yoy_lag4q", "entry_emp_yoy_lag8q",
]
out = out[[c for c in cols if c in out.columns]]

integrated_path = DATA_DIR / "firmscape_integrated_cbsa_quarterly.csv"
out.to_csv(integrated_path, index=False)
print(f"✓ Wrote integrated panel to {integrated_path}")
print(f"  Shape: {out.shape}")

✓ Wrote integrated panel to /Users/rishil/Desktop/FirmScape/data/firmscape_integrated_cbsa_quarterly.csv
  Shape: (83230, 44)


## Step 10: Clean the Integrated Panel

In [ ]:
# Load integrated panel for cleaning
df = pd.read_csv(integrated_path)

# Coerce year/quarter to Int64
for col in ("year", "quarter"):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Parse yq into date column
if "yq" in df.columns:
    df["yq"] = df["yq"].astype(str)
    per = pd.PeriodIndex(df["yq"].str.replace("Q", "Q", regex=False), freq="Q")
    ts = per.to_timestamp(how="end")
    df["date"] = ts.strftime("%Y-%m-%d 23:59:59.999999999")
elif "year" in df.columns and "quarter" in df.columns:
    per = pd.PeriodIndex(
        df["year"].astype("Int64").astype(str)
        + "Q"
        + df["quarter"].astype("Int64").astype(str),
        freq="Q",
    )
    ts = per.to_timestamp(how="end")
    df["date"] = ts.strftime("%Y-%m-%d 23:59:59.999999999")

# Drop rows where all core signal columns are missing
core_cols = [
    "fhfa_index", "zillow_price_q", "fhfa_yoy", "zillow_yoy",
    "firms_founded", "firms_founded_yoy",
    "hhi_new", "top_industry_share_new", "industry_count_new",
    "entry_emp_proxy", "entry_emp_yoy",
]
existing_core = [c for c in core_cols if c in df.columns]
if existing_core:
    df = df.dropna(subset=existing_core, how="all")

# Sort
sort_cols = [c for c in ("cbsa_code", "year", "quarter") if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols)

# Column order to match reference exactly
ref_cols = [
    "cbsa_code", "metro_name", "city_state", "year", "quarter",
    "fhfa_index", "zillow_price_q", "fhfa_qoq", "fhfa_yoy", "zillow_qoq", "zillow_yoy",
    "firms_founded", "firms_founded_yoy", "hhi_new", "top_industry_share_new", "industry_count_new",
    "firm_count_total", "hhi_stock", "top_industry_share_stock", "industry_count_stock",
    "share_large_proxy", "median_size_mid",
    "firms_founded_yoy_lag1q", "hhi_new_lag1q", "firms_founded_yoy_lag2q", "hhi_new_lag2q",
    "firms_founded_yoy_lag4q", "hhi_new_lag4q", "firms_founded_yoy_lag8q", "hhi_new_lag8q",
    "fhfa_yoy_fwd4q", "zillow_yoy_fwd4q", "yq",
    "date",
]
cols = [c for c in ref_cols if c in df.columns]
df = df[cols]

cleaned_path = DATA_DIR / "firmscape_integrated_cbsa_quarterly_cleaned.csv"
df.to_csv(cleaned_path, index=False)

print(f"✓ Wrote cleaned panel to {cleaned_path}")
print(f"  Shape: {df.shape}")
df.head()

✓ Wrote cleaned panel to /Users/rishil/Desktop/FirmScape/data/firmscape_integrated_cbsa_quarterly_cleaned.csv
  Shape: (70819, 34)


,cbsa_code,metro_name,city_state,year,quarter,fhfa_index,zillow_price_q,fhfa_qoq,fhfa_yoy,zillow_qoq,...,firms_founded_yoy_lag2q,hhi_new_lag2q,firms_founded_yoy_lag4q,hhi_new_lag4q,firms_founded_yoy_lag8q,hhi_new_lag8q,fhfa_yoy_fwd4q,zillow_yoy_fwd4q,yq,date
8,10180,"Abilene, TX","Abilene, TX",1977,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1977Q1,1977-03-31 23:59:59.999999999
20,10180,"Abilene, TX","Abilene, TX",1980,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1980Q1,1980-03-31 23:59:59.999999999
24,10180,"Abilene, TX","Abilene, TX",1981,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.5,1.0,NaN,NaN,NaN,NaN,1981Q1,1981-03-31 23:59:59.999999999
32,10180,"Abilene, TX","Abilene, TX",1983,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,0.5,NaN,NaN,1983Q1,1983-03-31 23:59:59.999999999
36,10180,"Abilene, TX","Abilene, TX",1984,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,0.5,NaN,NaN,NaN,NaN,1984Q1,1984-03-31 23:59:59.999999999
